In [ ]:
import sys
sys.path.append("..")

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE

from src.preprocessing import load_data, split_data, build_preprocessor, IQRCapper, TARGET

sns.set_style("whitegrid")

In [ ]:
raw = pd.read_csv("../data/raw/student_mental_health.csv")

print("Shape:", raw.shape)
print("Duplicated rows:", raw.duplicated().sum())
print("Duplicated student ids:", raw["Student_ID"].duplicated().sum())
print("Missing targets:", raw[TARGET].isnull().sum())

In [ ]:
df = load_data("../data/raw/student_mental_health.csv")
print("Shape after cleaning:", df.shape)
df.head()

In [ ]:
X_train, X_test, y_train, y_test = split_data(df)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

In [ ]:
print("Train distribution (%)")
print((y_train.value_counts(normalize=True) * 100).round(2))
print()
print("Test distribution (%)")
print((y_test.value_counts(normalize=True) * 100).round(2))

## 4. Numerical and categorical columns

I separate the columns by type because they need different transformations.

In [ ]:
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()

print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

## 5. Missing values

I look at the missing values in the training set, and I will fill the numerical ones with the median and the categorical ones with the most frequent value.

In [ ]:
missing = X_train.isnull().sum()
print(missing[missing > 0])
print("Total:", missing.sum())

In [ ]:
if missing.sum() > 0:
    plt.figure(figsize=(8, 4))
    missing[missing > 0].plot(kind="bar", color="salmon")
    plt.title("Missing values in the training set")
    plt.xlabel("Column")
    plt.ylabel("Number of missing values")
    plt.xticks(rotation=45)
    plt.show()

## 6. Outliers

I cap the extreme values with the IQR method, using only the training data to compute the limits.

In [ ]:
filled = X_train[num_cols].fillna(X_train[num_cols].median())

capper = IQRCapper()
capped = pd.DataFrame(capper.fit_transform(filled), columns=num_cols)

changed = (filled.values != capped.values).sum(axis=0)
outliers_count = pd.Series(changed, index=num_cols)
print(outliers_count)

In [ ]:
col = outliers_count.idxmax()

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
sns.boxplot(x=filled[col], color="salmon")
plt.title(col + " before capping")
plt.xlabel(col)

plt.subplot(1, 2, 2)
sns.boxplot(x=capped[col], color="lightgreen")
plt.title(col + " after capping")
plt.xlabel(col)

plt.tight_layout()
plt.show()

## 7. Encoding

The models only understand numbers, so the categorical columns are transformed with one-hot encoding.

In [ ]:
print(X_train[cat_cols].nunique())
print()
for col in cat_cols:
    print(col, ":", X_train[col].dropna().unique())

## 8. Scaling and full preprocessing

I put imputation, capping, scaling and encoding in one `ColumnTransformer`, then fit it on the train set only and apply it to the test set.

In [ ]:
preprocessor = build_preprocessor(X_train)

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

print("Train shape:", X_train_prep.shape)
print("Test shape:", X_test_prep.shape)

In [ ]:
feature_names = preprocessor.get_feature_names_out()
train_prep_df = pd.DataFrame(X_train_prep, columns=feature_names)

print("Missing values after preprocessing:", train_prep_df.isnull().sum().sum())
train_prep_df.head()

In [ ]:
num_features = [c for c in feature_names if c.startswith("num__")]
train_prep_df[num_features].describe().loc[["mean", "std"]].round(2)

## 9. Class imbalance

I compute the ratio between the biggest and the smallest class to decide if SMOTE is needed.

In [ ]:
counts = y_train.value_counts()
print(counts)
print("Ratio majority / minority:", round(counts.max() / counts.min(), 2))

SMOTE creates new examples of the minority classes, and it must only be applied on the training set.

In [ ]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_prep, y_train)

print("Before SMOTE:")
print(y_train.value_counts())
print()
print("After SMOTE:")
print(pd.Series(y_train_smote).value_counts())

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
sns.countplot(x=y_train, color="steelblue")
plt.title("Classes before SMOTE")
plt.xlabel(TARGET)
plt.ylabel("Number of students")

plt.subplot(1, 2, 2)
sns.countplot(x=y_train_smote, color="seagreen")
plt.title("Classes after SMOTE")
plt.xlabel(TARGET)
plt.ylabel("Number of students")

plt.tight_layout()
plt.show()

In the final pipeline SMOTE is a step before the model, so it is only applied on the training folds and never on the test data.

## 10. Saving the train and test sets

I save the split data in `data/processed` so the next notebooks use exactly the same train and test sets.

In [ ]:
os.makedirs("../data/processed", exist_ok=True)

X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

print(os.listdir("../data/processed"))

## 11. Cleaning decisions

| Problem | Decision | Why |
|---|---|---|
| Duplicated rows | Removed | They give the same information twice |
| Student_ID column | Dropped | It is only an identifier, not a feature |
| Rows without target | Removed | They can not be used for supervised learning |
| Missing numerical values | Median | The median is not affected by extreme values |
| Missing categorical values | Most frequent value | Keeps the most common category without removing rows |
| Outliers | Capped with IQR | Keeps all the rows and reduces the effect of extreme values |
| Categorical variables | One-hot encoding | Models need numbers and there is no wrong order between categories |
| Numerical variables | StandardScaler | Puts all the variables on the same scale |
| Class imbalance | SMOTE on the training set only | Helps the models learn the minority classes without touching the test set |
| Data leakage | Everything is fitted on train only inside a pipeline | The test set stays unseen |
